# ❄️🐉 cryoDRGN on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ts387/cryodrgn/blob/claude/cryodrgn-colab-notebook-5mf3p3/cryoDRGN_colab.ipynb)

**cryoDRGN** is a neural-network method for **heterogeneous cryo-EM reconstruction** — it learns a *continuous* distribution of 3D structures directly from a single-particle dataset.

This notebook walks you end-to-end on a free/Pro Colab GPU:

| Step | What happens |
|------|--------------|
| 1. Setup | Check the GPU, install cryoDRGN, mount Google&nbsp;Drive |
| 2. Inputs | Point to your particles, poses and CTF (sourced from Drive) |
| 3. Preprocess | `downsample` → `parse_pose_*` → `parse_ctf_*` |
| 4. Sanity check | `backproject_voxel` a subset and view slices |
| 5. Train | `train_vae` a heterogeneous model |
| 6. Analyze | `analyze` the latent space + view plots and volumes inline |
| 7. Save | Sync results back to your Google Drive |

> **You will need**, from an upstream consensus refinement (RELION or cryoSPARC):
> - a **particle stack** — `.mrcs` / `.star` / `.cs` / `.txt`
> - a **`.star`** (RELION) **or `.cs`** (cryoSPARC) file to extract **poses** and **CTF** from
>
> Don't have data yet? Try the tutorial dataset (EMPIAR-10076) from the
> [cryoDRGN user guide](https://ez-lab.gitbook.io/cryodrgn/).

📖 Docs: <https://ez-lab.gitbook.io/cryodrgn/> · 💻 GitHub: <https://github.com/ml-struct-bio/cryodrgn> · 📄 [Zhong et al., *Nature Methods* 2021](https://doi.org/10.1038/s41592-020-01049-4)

---
### ⚙️ Before you start — turn on the GPU
**Runtime → Change runtime type → Hardware accelerator → GPU** (a T4 is fine for `D=128`; use an A100/L4 on Colab Pro for `D=256`).

Then run the cells **in order** (▶ on each, or *Runtime → Run all*). Each cell is a collapsible **form** — edit the fields on the right, no coding required.

## 1 · Setup

In [ ]:
#@title 1.1 · Check the GPU runtime { display-mode: "form" }
#@markdown Confirms a CUDA GPU is attached. If this prints **"No GPU found"**, go to
#@markdown **Runtime → Change runtime type → GPU** and re-run this cell.
import subprocess, sys

print("=" * 60)
gpu = subprocess.run(["nvidia-smi",
                      "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode == 0 and gpu.stdout.strip():
    name, mem, driver = [x.strip() for x in gpu.stdout.strip().split(",")]
    print(f"✅ GPU detected : {name}")
    print(f"   Memory       : {mem}")
    print(f"   Driver       : {driver}")
else:
    print("❌ No GPU found!")
    print("   Runtime → Change runtime type → Hardware accelerator → GPU,")
    print("   then re-run this cell. cryoDRGN training needs a GPU.")
print("=" * 60)

In [ ]:
#@title 1.2 · Install cryoDRGN { display-mode: "form" }
#@markdown Installs cryoDRGN from PyPI. Colab's pre-installed PyTorch/CUDA are kept.
#@markdown <br>• **stable** – the recommended release &nbsp;•&nbsp; **beta** – newest dev build from TestPyPI
release_channel = "stable"  #@param ["stable", "beta"]
#@markdown Optionally pin an exact version (e.g. `4.3.0`); leave blank for the latest.
version = ""  #@param {type:"string"}
#@markdown A few dependencies are pinned to versions other than Colab's defaults, so the
#@markdown runtime **restarts automatically** at the end. That is expected — just carry
#@markdown on with the next cell afterwards.
restart_after_install = True  #@param {type:"boolean"}

import subprocess, sys

pkg = "cryodrgn"
if version.strip():
    pkg = f"cryodrgn=={version.strip()}"

if release_channel == "beta":
    cmd = [sys.executable, "-m", "pip", "install", "-q",
           "-i", "https://test.pypi.org/simple/",
           "--extra-index-url", "https://pypi.org/simple/",
           "cryodrgn", "--pre"]
    if version.strip():
        cmd[cmd.index("cryodrgn")] = pkg
else:
    cmd = [sys.executable, "-m", "pip", "install", "-q", pkg]

print("Installing", pkg, f"({release_channel} channel) — this takes ~1-2 min...\n")
ret = subprocess.run(cmd)
if ret.returncode != 0:
    raise SystemExit("❌ pip install failed — see the log above.")

print("\n✅ cryoDRGN installed.")
if restart_after_install:
    print("🔄 Restarting the runtime to finalize the install (this is normal)...")
    print("   When it reconnects, continue from cell 1.3 — do NOT re-run this cell.")
    get_ipython().kernel.do_shutdown(True)

In [ ]:
#@title 1.3 · Verify the installation { display-mode: "form" }
#@markdown Run this **after** the runtime has restarted.
import torch, cryodrgn

print(f"cryoDRGN version : {cryodrgn.__version__}")
print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device      : {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  CUDA not available — check that the GPU runtime is selected (cell 1.1).")

# quick smoke-test of the command-line entry point
import subprocess
print("\n$ cryodrgn --version")
print(subprocess.run(["cryodrgn", "--version"], capture_output=True, text=True).stdout.strip())

## 2 · Connect Google Drive

We use **two locations**, which is standard practice for cryo-EM on Colab:

- 📁 **Drive project folder** — *durable* storage for your inputs and final results. Survives disconnects.
- ⚡ **Local scratch** (`/content/...`) — *fast* disk for the downsampled stack that training reads. Wiped when the runtime ends.

We read inputs from Drive and downsample onto the fast local scratch; training then writes its model **directly to Drive** (so per-epoch checkpoints survive a disconnect), and a final step syncs any remaining local artifacts back (Step 9).

In [ ]:
#@title 2.1 · Mount Google Drive { display-mode: "form" }
#@markdown Click the link that appears, pick your Google account, and paste the code
#@markdown (or approve the pop-up). Your Drive appears under `/content/drive/MyDrive`.
from google.colab import drive
drive.mount("/content/drive")
print("\n✅ Drive mounted at /content/drive/MyDrive")

In [ ]:
#@title 2.2 · Choose your project folder { display-mode: "form" }
#@markdown **`drive_project_dir`** — a folder in *your* Drive for this project (created if missing).
#@markdown Put your input files here, and final results are saved back here.
drive_project_dir = "/content/drive/MyDrive/cryodrgn_project"  #@param {type:"string"}
#@markdown **`local_work_dir`** — fast local scratch where training runs.
local_work_dir = "/content/cryodrgn_work"  #@param {type:"string"}

import os

DRIVE_DIR = os.path.abspath(drive_project_dir)
WORK_DIR = os.path.abspath(local_work_dir)
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)

# Persist these across cells for the rest of the notebook.
os.environ["CRYODRGN_DRIVE_DIR"] = DRIVE_DIR
os.environ["CRYODRGN_WORK_DIR"] = WORK_DIR
os.chdir(WORK_DIR)

print(f"📁 Drive project (durable) : {DRIVE_DIR}")
print(f"⚡ Local scratch (fast)    : {WORK_DIR}")
print(f"📂 Working directory       : {os.getcwd()}")
print("\nContents of your Drive project folder:")
for f in sorted(os.listdir(DRIVE_DIR)) or ["(empty — upload your inputs here)"]:
    print("   ", f)

## 3 · Point to your input files

cryoDRGN needs three things, all derived from an upstream **consensus refinement**:

1. **Particle images** — the stack you refined (`.mrcs`, `.star`, `.cs`, or a `.txt` of `.mrcs` paths).
2. **Poses** — orientation + shift per particle, extracted from the refinement's `.star`/`.cs`.
3. **CTF parameters** — extracted from the same `.star`/`.cs`.

Fill in the paths below (they usually live inside your Drive project folder from Step 2).

In [ ]:
#@title 3.1 · Locate inputs on Drive { display-mode: "form" }
import os
DRIVE_DIR = os.environ["CRYODRGN_DRIVE_DIR"]
WORK_DIR = os.environ["CRYODRGN_WORK_DIR"]

#@markdown **Particle stack** — path to your images (`.mrcs`/`.star`/`.cs`/`.txt`).
particles = "/content/drive/MyDrive/cryodrgn_project/particles.mrcs"  #@param {type:"string"}

#@markdown **Metadata file** — the RELION `.star` **or** cryoSPARC `.cs` file that holds poses & CTF.
metadata_file = "/content/drive/MyDrive/cryodrgn_project/particles.star"  #@param {type:"string"}

#@markdown **`datadir`** *(optional)* — folder holding the `.mrcs` referenced by a `.star`/`.cs`,
#@markdown used only if the paths inside it are broken. Leave blank if not needed.
datadir = ""  #@param {type:"string"}

# make these available to later cells
os.environ["CRYODRGN_PARTICLES"] = particles
os.environ["CRYODRGN_META"] = metadata_file
os.environ["CRYODRGN_DATADIR"] = datadir

print("Checking inputs...\n")
for label, path in [("Particles", particles), ("Metadata (.star/.cs)", metadata_file)]:
    ok = os.path.exists(path)
    print(f"  {'✅' if ok else '❌'} {label}: {path}")
    if not ok:
        print("       ^ not found — fix the path above (check spelling / that it's in your Drive).")
if datadir:
    print(f"  {'✅' if os.path.isdir(datadir) else '❌'} datadir: {datadir}")

## 4 · Preprocess

Three quick commands turn your raw inputs into what `train_vae` expects:
`downsample` the images, then extract `pose.pkl` and `ctf.pkl`.

In [ ]:
#@title 4.1 · Downsample the images { display-mode: "form" }
#@markdown Reads your particles from Drive and writes a smaller stack to **fast local disk**
#@markdown (this is what training reads). Smaller boxes train **much** faster — start at **128**
#@markdown to sanity-check and filter, then optionally redo at **256**. Max recommended is 256.
box_size = 128  #@param [64, 128, 256] {type:"raw"}
#@markdown Split output into chunks of this many images if you hit memory limits (`0` = off).
chunk = 0  #@param {type:"integer"}

import os
particles = os.environ["CRYODRGN_PARTICLES"]
datadir = os.environ.get("CRYODRGN_DATADIR", "")
WORK_DIR = os.environ["CRYODRGN_WORK_DIR"]
out_mrcs = os.path.join(WORK_DIR, f"particles.{box_size}.mrcs")

cmd = f'cryodrgn downsample "{particles}" -D {box_size} -o "{out_mrcs}"'
if chunk and int(chunk) > 0:
    cmd += f" --chunk {int(chunk)}"
if datadir:
    cmd += f' --datadir "{datadir}"'

print("$", cmd, "\n")
get_ipython().system(cmd)

# With --chunk, images are split across particles.<D>.<i>.mrcs tiles indexed by a
# particles.<D>.txt file — that .txt (not the .mrcs) is what later steps must read.
stack = os.path.splitext(out_mrcs)[0] + ".txt" if (chunk and int(chunk) > 0) else out_mrcs
os.environ["CRYODRGN_DOWNSAMPLED"] = stack
print(f"\n✅ Downsampled stack → {stack}")

In [ ]:
#@title 4.2 · Parse poses  →  pose.pkl { display-mode: "form" }
#@markdown Choose the format of your metadata file.
pose_source = "RELION .star"  #@param ["RELION .star", "cryoSPARC .cs"]
#@markdown **`box_size_D`** — box size of the **consensus refinement** (the *original*, un-downsampled
#@markdown images). Required for cryoSPARC `.cs`; for `.star` it is auto-detected if present, so
#@markdown you can leave it at `0`.
box_size_D = 0  #@param {type:"integer"}

import os
meta = os.environ["CRYODRGN_META"]
WORK_DIR = os.environ["CRYODRGN_WORK_DIR"]
pose_pkl = os.path.join(WORK_DIR, "pose.pkl")
os.environ["CRYODRGN_POSE"] = pose_pkl

if pose_source == "RELION .star":
    cmd = f'cryodrgn parse_pose_star "{meta}" -o "{pose_pkl}"'
    if int(box_size_D) > 0:
        cmd += f" -D {int(box_size_D)}"
else:
    if int(box_size_D) <= 0:
        print("⚠️  cryoSPARC .cs usually needs -D (the consensus box size). "
              "If parsing fails, set box_size_D above.")
    cmd = f'cryodrgn parse_pose_csparc "{meta}" -o "{pose_pkl}"'
    if int(box_size_D) > 0:
        cmd += f" -D {int(box_size_D)}"

print("$", cmd, "\n")
get_ipython().system(cmd)
print(f"\n✅ Poses → {pose_pkl}")

In [ ]:
#@title 4.3 · Parse CTF  →  ctf.pkl { display-mode: "form" }
ctf_source = "RELION .star"  #@param ["RELION .star", "cryoSPARC .cs"]
#@markdown For `.star` files, `-D` (box) and `--Apix` are auto-read when present.
#@markdown Fill these in only if they are missing from the file (`0` = auto).
box_size_D = 0  #@param {type:"integer"}
apix = 0  #@param {type:"number"}

import os
meta = os.environ["CRYODRGN_META"]
WORK_DIR = os.environ["CRYODRGN_WORK_DIR"]
ctf_pkl = os.path.join(WORK_DIR, "ctf.pkl")
os.environ["CRYODRGN_CTF"] = ctf_pkl

if ctf_source == "RELION .star":
    cmd = f'cryodrgn parse_ctf_star "{meta}" -o "{ctf_pkl}"'
    if int(box_size_D) > 0:
        cmd += f" -D {int(box_size_D)}"
    if float(apix) > 0:
        cmd += f" --Apix {apix}"
else:
    cmd = f'cryodrgn parse_ctf_csparc "{meta}" -o "{ctf_pkl}"'

print("$", cmd, "\n")
get_ipython().system(cmd)
print(f"\n✅ CTF → {ctf_pkl}")

## 5 · (Optional) Sanity-check poses & CTF

Before spending GPU time on training, back-project a subset of particles into a 3D map.
It should look like a **low-resolution version of your consensus structure**. If it's noise,
the poses/CTF are probably mis-parsed (a common fix is toggling `uninvert_data`).

In [ ]:
#@title 5.1 · Voxel back-projection of a subset { display-mode: "form" }
#@markdown Number of particles to use (fewer = faster, noisier).
n_particles = 10000  #@param {type:"integer"}
#@markdown Tick if your particles are dark-on-light (flips the data sign).
uninvert_data = False  #@param {type:"boolean"}

import os
ds = os.environ["CRYODRGN_DOWNSAMPLED"]
pose = os.environ["CRYODRGN_POSE"]
ctf = os.environ["CRYODRGN_CTF"]
WORK_DIR = os.environ["CRYODRGN_WORK_DIR"]
bp_dir = os.path.join(WORK_DIR, "backproject")
os.environ["CRYODRGN_BACKPROJECT"] = bp_dir

cmd = (f'cryodrgn backproject_voxel "{ds}" --poses "{pose}" --ctf "{ctf}" '
       f'-o "{bp_dir}" --first {int(n_particles)}')
if uninvert_data:
    cmd += " --uninvert-data"

print("$", cmd, "\n")
get_ipython().system(cmd)
print(f"\n✅ Map → {bp_dir}/backproject.mrc")

In [ ]:
#@title 5.2 · View central slices of the back-projected map { display-mode: "form" }
import os, glob
import numpy as np
import matplotlib.pyplot as plt
from cryodrgn.mrcfile import parse_mrc

bp_dir = os.environ["CRYODRGN_BACKPROJECT"]
hits = glob.glob(os.path.join(bp_dir, "*.mrc"))
if not hits:
    raise FileNotFoundError(f"No .mrc found in {bp_dir} — run cell 5.1 first.")

vol, _ = parse_mrc(hits[0])
D = vol.shape[0]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (axis, title) in zip(axes, [(0, "Z"), (1, "Y"), (2, "X")]):
    sl = vol.take(D // 2, axis=axis)
    ax.imshow(sl, cmap="Greys_r")
    ax.set_title(f"central {title} slice")
    ax.axis("off")
fig.suptitle(os.path.basename(hits[0]))
plt.tight_layout()
plt.show()
print("Looks like your structure? ✅ Proceed to training.\n"
      "Just noise? ❌ Re-check poses/CTF and try toggling `uninvert_data` in 5.1.")

## 6 · Train the cryoDRGN model

Now train the VAE for heterogeneous reconstruction. Model outputs (per-epoch `weights.*.pkl`,
`z.*.pkl`, `config.yaml`) are written **directly to your Drive project folder**, so they survive
a disconnect and you can resume with `--load`.

**Tips:** `zdim` 8 is a good default (use 1 for a single motion axis, ≥10 for complex mixtures).
For a first pass, 25 epochs at `D=128` is typical. On a T4, `D=128` runs roughly a few minutes/epoch
for ~100k particles; `D=256` is far slower — prefer Colab Pro.

In [ ]:
#@title 6.1 · Configure & launch train_vae { display-mode: "form" }
#@markdown **Latent dimension** — size of the conformational latent space.
zdim = 8  #@param [1, 2, 4, 8, 10] {type:"raw"}
#@markdown **Epochs** — full passes over the dataset.
num_epochs = 25  #@param {type:"integer"}
#@markdown **Batch size** — increase to better use a big GPU (affects dynamics).
batch_size = 8  #@param [8, 16, 32] {type:"raw"}
#@markdown **Output folder name** (created inside your Drive project folder).
output_name = "00_cryodrgn128"  #@param {type:"string"}
#@markdown Dark-on-light particles? (must match what worked in Step 5)
uninvert_data = False  #@param {type:"boolean"}
#@markdown Lazy loading — stream images from disk if the stack is too big for RAM.
lazy = False  #@param {type:"boolean"}

import os
ds = os.environ["CRYODRGN_DOWNSAMPLED"]
pose = os.environ["CRYODRGN_POSE"]
ctf = os.environ["CRYODRGN_CTF"]
DRIVE_DIR = os.environ["CRYODRGN_DRIVE_DIR"]
outdir = os.path.join(DRIVE_DIR, output_name)
os.environ["CRYODRGN_OUTDIR"] = outdir

cmd = (f'cryodrgn train_vae "{ds}" --poses "{pose}" --ctf "{ctf}" '
       f'--zdim {zdim} -n {int(num_epochs)} -b {batch_size} -o "{outdir}"')
if uninvert_data:
    cmd += " --uninvert-data"
if lazy:
    cmd += " --lazy"

print("$", cmd, "\n" + "=" * 70)
get_ipython().system(cmd)
print("=" * 70 + f"\n✅ Training complete. Model saved to: {outdir}")

In [ ]:
#@title 6.2 · (Optional) Resume / extend training { display-mode: "form" }
#@markdown Continue an interrupted or finished run for more epochs. Point `resume_epoch`
#@markdown at the last saved checkpoint (e.g. `25` loads `weights.25.pkl`) and set a larger
#@markdown `num_epochs`. Uses the same output folder from 6.1.
resume_epoch = 25  #@param {type:"integer"}
num_epochs = 50  #@param {type:"integer"}

import os
ds = os.environ["CRYODRGN_DOWNSAMPLED"]
pose = os.environ["CRYODRGN_POSE"]
ctf = os.environ["CRYODRGN_CTF"]
outdir = os.environ["CRYODRGN_OUTDIR"]
weights = os.path.join(outdir, f"weights.{int(resume_epoch)}.pkl")

if not os.path.exists(weights):
    raise FileNotFoundError(f"{weights} not found — check resume_epoch / that 6.1 has run.")

# reuse the same zdim/batch the run was created with (read from config.yaml)
import yaml
cfg = yaml.safe_load(open(os.path.join(outdir, "config.yaml")))
zdim = cfg["model_args"]["zdim"]

cmd = (f'cryodrgn train_vae "{ds}" --poses "{pose}" --ctf "{ctf}" '
       f'--zdim {zdim} -n {int(num_epochs)} -o "{outdir}" --load "{weights}"')
print("$", cmd, "\n" + "=" * 70)
get_ipython().system(cmd)
print("=" * 70 + "\n✅ Done.")

## 7 · Analyze the results

`cryodrgn analyze` visualizes the latent space (PCA + UMAP), then generates representative
volumes by k-means-sampling the latent space and traversing its principal components.

In [ ]:
#@title 7.1 · Run cryodrgn analyze { display-mode: "form" }
#@markdown Epoch to analyze — leave at **-1** to auto-pick the latest saved epoch.
epoch = -1  #@param {type:"integer"}
#@markdown Number of k-means volumes to generate.
ksample = 20  #@param {type:"integer"}
#@markdown Pixel size (Å/px) written into volume headers (`0` = read from ctf.pkl / default 1).
apix = 0  #@param {type:"number"}

import os, re, glob
outdir = os.environ["CRYODRGN_OUTDIR"]

if int(epoch) < 0:  # auto-detect latest z.N.pkl
    epochs = []
    for p in glob.glob(os.path.join(outdir, "z.*.pkl")):
        m = re.search(r"z\.(\d+)\.pkl$", os.path.basename(p))
        if m:
            epochs.append(int(m.group(1)))
    if not epochs:
        raise FileNotFoundError(f"No z.N.pkl checkpoints in {outdir} — has 6.1 finished?")
    epoch = max(epochs)
    print(f"Auto-selected latest epoch: {epoch}")

os.environ["CRYODRGN_EPOCH"] = str(int(epoch))
cmd = f'cryodrgn analyze "{outdir}" {int(epoch)} --ksample {int(ksample)}'
if float(apix) > 0:
    cmd += f" --Apix {apix}"

print("$", cmd, "\n" + "=" * 70)
get_ipython().system(cmd)
print("=" * 70 + f"\n✅ Analysis → {outdir}/analyze.{int(epoch)}")

In [ ]:
#@title 7.2 · View latent-space plots inline { display-mode: "form" }
import os, glob
from IPython.display import Image, display, Markdown

outdir = os.environ["CRYODRGN_OUTDIR"]
epoch = os.environ["CRYODRGN_EPOCH"]
adir = os.path.join(outdir, f"analyze.{epoch}")

for fname, caption in [
    ("z_pca.png", "**PCA** of the latent embeddings (colored by k-means cluster)"),
    ("umap.png", "**UMAP** of the latent embeddings"),
    ("z_pca_marginals.png", "PCA with marginal distributions"),
    ("umap_marginals.png", "UMAP with marginal distributions"),
    (f"learning_curve_epoch{epoch}.png", "Training loss curve"),
]:
    path = os.path.join(adir, fname)
    if os.path.exists(path):
        display(Markdown(caption))
        display(Image(path, width=520))
    else:
        # k-means subfolder holds some variants
        alt = glob.glob(os.path.join(adir, "kmeans*", fname))
        if alt:
            display(Markdown(caption))
            display(Image(alt[0], width=520))

In [ ]:
#@title 7.3 · Interactive 3D view of a generated volume { display-mode: "form" }
#@markdown Renders one of the k-means representative maps as a 3D isosurface you can rotate.
#@markdown Change `volume_index` (0 … ksample-1) to inspect different structures.
volume_index = 0  #@param {type:"integer"}
#@markdown Isosurface threshold as a percentile of density (higher = tighter surface).
iso_percentile = 99.0  #@param {type:"slider", min:90, max:99.9, step:0.1}
#@markdown Downsample the box for a snappier render.
display_box = 64  #@param [48, 64, 96] {type:"raw"}

import os, glob
import numpy as np
import plotly.graph_objects as go
from cryodrgn.mrcfile import parse_mrc

outdir = os.environ["CRYODRGN_OUTDIR"]
epoch = os.environ["CRYODRGN_EPOCH"]
adir = os.path.join(outdir, f"analyze.{epoch}")

vols = sorted(glob.glob(os.path.join(adir, "kmeans*", "vol_*.mrc")))
if not vols:
    raise FileNotFoundError(f"No k-means volumes in {adir} — run 7.1 without --skip-vol.")
vol_path = vols[int(volume_index) % len(vols)]
vol, _ = parse_mrc(vol_path)

# light box downsampling for display
D = vol.shape[0]
if D > int(display_box):
    step = int(round(D / int(display_box)))
    vol = vol[::step, ::step, ::step]
D = vol.shape[0]

x, y, z = np.mgrid[0:D, 0:D, 0:D]
iso = float(np.percentile(vol, iso_percentile))
fig = go.Figure(go.Isosurface(
    x=x.flatten(), y=y.flatten(), z=z.flatten(), value=vol.flatten(),
    isomin=iso, isomax=float(vol.max()),
    surface_count=1, colorscale="Greys", showscale=False, caps=dict(x_show=False, y_show=False, z_show=False),
))
fig.update_layout(title=os.path.basename(vol_path), width=560, height=560,
                  scene=dict(xaxis_visible=False, yaxis_visible=False, zaxis_visible=False))
fig.show()
print(f"Showing {vol_path}  ({len(vols)} volumes available; set volume_index 0..{len(vols)-1})")

## 8 · (Optional) Generate a volume at a chosen latent value

Use `eval_vol` to decode a structure at any point `z` in latent space — for example a cluster
center read from the plots above, or a custom coordinate. The number of values you give `-z`
must equal your `zdim`.

In [ ]:
#@title 8.1 · Decode a volume at a specific z { display-mode: "form" }
#@markdown Space-separated latent coordinate, length == `zdim` (e.g. `0.5 -1.2 0 0 0 0 0 0`).
z_value = "0 0 0 0 0 0 0 0"  #@param {type:"string"}
#@markdown Output filename (saved in your Drive project folder).
output_name = "my_volume.mrc"  #@param {type:"string"}
#@markdown Pixel size (Å/px) for the header (`0` = default 1).
apix = 0  #@param {type:"number"}

import os
outdir = os.environ["CRYODRGN_OUTDIR"]
DRIVE_DIR = os.environ["CRYODRGN_DRIVE_DIR"]
weights = os.path.join(outdir, "weights.pkl")
config = os.path.join(outdir, "config.yaml")
out_mrc = os.path.join(DRIVE_DIR, output_name)

cmd = f'cryodrgn eval_vol "{weights}" --config "{config}" -z {z_value} -o "{out_mrc}"'
if float(apix) > 0:
    cmd += f" --Apix {apix}"

print("$", cmd, "\n")
get_ipython().system(cmd)
print(f"\n✅ Volume → {out_mrc}")

## 9 · Save & download results

Model outputs already live on Drive (Step 6 wrote there directly). Use these cells to copy any
extra local artifacts to Drive, or to download a results folder to your computer.

In [ ]:
#@title 9.1 · Sync local scratch artifacts to Drive { display-mode: "form" }
#@markdown Copies the parsed `pose.pkl` / `ctf.pkl` and the back-projection map from local
#@markdown scratch into your Drive project folder (the downsampled `.mrcs`, often several GB,
#@markdown is skipped by default since it is easily regenerated).
include_downsampled_stack = False  #@param {type:"boolean"}

import os, shutil
WORK_DIR = os.environ["CRYODRGN_WORK_DIR"]
DRIVE_DIR = os.environ["CRYODRGN_DRIVE_DIR"]

to_copy = ["pose.pkl", "ctf.pkl"]
for name in to_copy:
    src = os.path.join(WORK_DIR, name)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(DRIVE_DIR, name))
        print(f"✅ {name} → Drive")

bp = os.path.join(WORK_DIR, "backproject")
if os.path.isdir(bp):
    shutil.copytree(bp, os.path.join(DRIVE_DIR, "backproject"), dirs_exist_ok=True)
    print("✅ backproject/ → Drive")

if include_downsampled_stack:
    ds = os.environ.get("CRYODRGN_DOWNSAMPLED", "")
    if ds and os.path.exists(ds):
        print(f"Copying {os.path.basename(ds)} (may be large)...")
        shutil.copy2(ds, os.path.join(DRIVE_DIR, os.path.basename(ds)))
        print("✅ downsampled stack → Drive")

print(f"\nAll durable results are in: {DRIVE_DIR}")

In [ ]:
#@title 9.2 · (Optional) Zip an analysis folder and download it { display-mode: "form" }
#@markdown Bundles `analyze.<epoch>/` (plots + volumes) into a zip and downloads it to your computer.
import os, shutil
from google.colab import files

outdir = os.environ["CRYODRGN_OUTDIR"]
epoch = os.environ["CRYODRGN_EPOCH"]
adir = os.path.join(outdir, f"analyze.{epoch}")
if not os.path.isdir(adir):
    raise FileNotFoundError(f"{adir} not found — run Step 7 first.")

zip_base = os.path.join(os.environ["CRYODRGN_WORK_DIR"], f"analyze.{epoch}")
print("Zipping", adir, "...")
shutil.make_archive(zip_base, "zip", adir)
print("Starting download of", zip_base + ".zip")
files.download(zip_base + ".zip")

## 10 · Tips, troubleshooting & next steps

**Colab session limits**
- Free Colab disconnects after idle time and caps total runtime. Because per-epoch checkpoints
  are written to Drive, you can always resume with **cell 6.2** (`--load`).
- For big datasets or `D=256`, use **Colab Pro/Pro+** (A100/L4, longer sessions, more RAM).

**Common issues**
- *Back-projection / volumes look like noise* → poses or CTF likely mis-parsed. Re-check the box
  size `-D` (must be the **consensus** box, not the downsampled one) and toggle `uninvert_data`.
- *`CUDA out of memory`* → downsample to a smaller box (128), lower the batch size, or use a bigger GPU.
- *`.star`/`.cs` paths broken* → set **`datadir`** (cell 3.1) to the folder holding the `.mrcs`.
- *Slow training* → make sure you copied particles to local disk (cell 3.2) and are at `D=128`.
- *Out of disk on `/content`* → free space by deleting the local downsampled stack, or work at `D=128`.

**Going further** (all available as `cryodrgn ...` commands)
- `cryodrgn filter <workdir>` — interactively remove junk particles, then retrain on the good subset.
- `cryodrgn analyze_landscape` / `analyze_landscape_full` — automated conformational landscape analysis.
- `cryodrgn graph_traversal` + `eval_vol` — build long trajectories/movies through latent space.
- `cryodrgn abinit_homo` / `abinit_het` — *ab-initio* reconstruction (no consensus poses needed).
- **cryoDRGN-ET** — heterogeneous subtomogram averaging for cryo-ET.

📖 Full walkthroughs: <https://ez-lab.gitbook.io/cryodrgn/> · Questions/bugs → [GitHub issues](https://github.com/ml-struct-bio/cryodrgn/issues).

---
*Volumes are best inspected in [ChimeraX](https://www.cgl.ucsf.edu/chimerax) — download the `.mrc`
files from your Drive project folder and open them there for publication-quality figures.*